# Lab type: review
# Course: ML301 — Deep Learning with PyTorch
# Lesson: Regularisation and Generalisation
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
print('PyTorch version:', torch.__version__)

## Part 1: Dropout

In [ ]:
# Inverted dropout: surviving neurons are scaled by 1/(1-p) during training
# so expected activation magnitude is preserved across both modes.

torch.manual_seed(0)
x = torch.ones(1, 8)
drop = nn.Dropout(p=0.5)

# Training mode: roughly half the neurons zeroed, survivors scaled by 2.0
drop.train()
out_train = drop(x)
print('Training mode output:', out_train)
print('Non-zero values:', out_train[out_train != 0].tolist())

# Evaluation mode: all neurons pass through unchanged (no scaling, no zeroing)
drop.eval()
out_eval = drop(x)
print('\nEvaluation mode output:', out_eval)
print('Evaluation mode == all ones:', (out_eval == 1.0).all().item())

**Question 1:** During training, surviving neurons are scaled by `1 / (1 - p)`. Why is this scaling applied at training time (inverted dropout) rather than at inference time? What would happen to expected activation magnitude at inference if no scaling were applied anywhere?

*(Write your answer here.)*

**Question 2:** A model is trained for 20 epochs, then the validation loop runs without calling `model.eval()`. Dropout rate is 0.4. Describe the two problems this causes: one affecting metric reliability, one affecting reproducibility.

*(Write your answer here.)*

## Part 2: BatchNorm

In [ ]:
# BatchNorm maintains running_mean and running_var during training.
# Training mode: normalises using the current mini-batch statistics (mean, var).
# Evaluation mode: normalises using the accumulated running statistics.

torch.manual_seed(0)

# Synthetic: 100 training samples, 16 features
X_train = torch.randn(100, 16) * 3 + 5   # mean≈5, std≈3
X_val   = torch.randn(20,  16) * 3 + 5

bn = nn.BatchNorm1d(16)

# --- Training pass: accumulate running statistics ---
bn.train()
with torch.no_grad():
    for i in range(0, 100, 32):
        _ = bn(X_train[i:i+32])

print(f'running_mean (first 4): {bn.running_mean[:4].tolist()}')
print(f'running_var  (first 4): {bn.running_var[:4].tolist()}')

# --- Evaluation pass: use running statistics ---
bn.eval()
with torch.no_grad():
    out_val = bn(X_val)
print(f'\nVal output mean (approx 0): {out_val.mean().item():.4f}')
print(f'Val output std  (approx 1): {out_val.std().item():.4f}')

**Question 3:** The training loop in a colleague's script calls `model.eval()` at the start of each epoch (before both training and validation). Identify the two specific failures this causes and how each degrades training.

*(Write your answer here.)*

**Question 4:** During training with batch_size=4 and BatchNorm layers, training loss is noisy and validation accuracy is lower than expected. Why does a very small batch size cause BatchNorm to degrade, and what are two alternatives to consider?

*(Write your answer here.)*

## Part 3: AdamW and Weight Decay

In [ ]:
# AdamW decouples weight decay from the adaptive gradient update.
# Adam with weight_decay: decay is folded into the gradient before adaptive scaling,
# so larger-gradient parameters get proportionally less decay (not true L2).
# AdamW: decay applied directly to weights after the gradient step — true L2.

torch.manual_seed(0)

# Minimal MLP for demonstration
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Synthetic regression data
X = torch.randn(200, 20)
y = X[:, :5].sum(dim=1, keepdim=True) + 0.1 * torch.randn(200, 1)

dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)
criterion = nn.MSELoss()

# --- Train with AdamW ---
model = MLP()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

model.train()
for epoch in range(20):
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()

# Weight norms after training
w1_norm = model.fc1.weight.norm().item()
w2_norm = model.fc2.weight.norm().item()
print(f'fc1 weight norm (with AdamW wd=0.01): {w1_norm:.4f}')
print(f'fc2 weight norm (with AdamW wd=0.01): {w2_norm:.4f}')

model_eval = MLP()
# Reset random state for fair comparison
torch.manual_seed(0)
model_nowd = MLP()
opt_nowd   = torch.optim.AdamW(model_nowd.parameters(), lr=1e-3, weight_decay=0.0)
model_nowd.train()
for epoch in range(20):
    for Xb, yb in loader:
        opt_nowd.zero_grad()
        loss = criterion(model_nowd(Xb), yb)
        loss.backward()
        opt_nowd.step()
print(f'fc1 weight norm (no weight decay):     {model_nowd.fc1.weight.norm().item():.4f}')
print(f'fc2 weight norm (no weight decay):     {model_nowd.fc2.weight.norm().item():.4f}')

**Question 5:** `Adam(weight_decay=1e-2)` and `AdamW(weight_decay=1e-2)` are not equivalent. Describe exactly how Adam's coupling of weight decay with adaptive scaling differs from AdamW's decoupled approach. Which should you prefer for regularisation, and why?

*(Write your answer here.)*

**Question 6:** Should weight decay be applied to bias terms and BatchNorm parameters? What is the conventional practice, and what is the practical effect on model behaviour if you regularise these parameters?

*(Write your answer here.)*

## Part 4: Layer Ordering — Linear → BatchNorm → ReLU

In [ ]:
# Standard placement: Linear → BatchNorm → Activation
# BatchNorm normalises the pre-activation distribution, giving ReLU a consistent
# zero-centred input. This keeps gradients well-scaled and avoids dead neurons.

torch.manual_seed(0)
x = torch.randn(32, 20)

# Correct ordering
correct_block = nn.Sequential(
    nn.Linear(20, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
)

# Incorrect: BatchNorm applied after ReLU — normalises already-non-negative values,
# losing the centring benefit and partially defeating the purpose of normalisation.
incorrect_block = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.BatchNorm1d(64),
)

out_correct   = correct_block(x)
out_incorrect = incorrect_block(x)

print('Correct   (BN before ReLU) — mean: {:.4f}, std: {:.4f}'.format(
    out_correct.mean().item(), out_correct.std().item()))
print('Incorrect (BN after ReLU) — mean: {:.4f}, std: {:.4f}'.format(
    out_incorrect.mean().item(), out_incorrect.std().item()))
print()
# BN after ReLU normalises non-negative values: mean shifts upward
# confirming BN now operates on a truncated (non-centred) distribution
print('Fraction of zeros in incorrect output:', (out_incorrect == 0).float().mean().item())

**Question 7:** Some research papers place BatchNorm *after* the activation. What is the practical risk of that ordering versus the standard Linear → BatchNorm → ReLU? When might post-activation normalisation still be acceptable?

*(Write your answer here.)*

## Summary

> **Final check:** Answer in one sentence each.

1. What does inverted dropout scaling achieve, and when is the scale factor applied?
2. Which two BatchNorm failure modes result from forgetting to switch between train and eval mode?
3. In one sentence, why should you prefer AdamW over Adam when weight decay regularisation matters?
4. What is the correct layer ordering for a Linear block with BatchNorm and ReLU, and why?